# 03 — Carregar as bases da Prefeitura (SIRF, lotes, limite municipal e cadastro)

**Objetivo:** ler o SIRF (sem dados pessoais), os lotes (um ponto garantido dentro de cada lote), o limite municipal e o cadastro municipal.

**Entradas:** `01_dados_brutos/sirf/`, `01_dados_brutos/geosanja/lotes.gpkg` e `limite_municipal.gpkg` (exportados do banco pelo QGIS), `01_dados_brutos/cadastro_municipal/cadastro_v2.xlsx`; `df_estab` (01).

**Saídas:** `03_processamento/03_carregar_bases_prefeitura/`: `df_sirf`, `df_lotes`, `df_lotes_descartados`, `df_limite`, `df_cadastro` (Parquet + manifesto).

**Como rodar:** de cima para baixo. Usa o cache quando ele vale. **Dados internos da Prefeitura:** nada daqui vai para o Git.

**Seções:** 1. SIRF · 2. Lotes · 3. Limite municipal · 4. Cadastro municipal · 5. Validação · 6. Resumo

In [ ]:
# ===================================================================
# PREPARAÇÃO: bibliotecas, config.yaml, log e entradas
# ===================================================================
# Bibliotecas que já vêm com o Python.
import datetime          # datas e horas
import importlib.util    # conferir se uma biblioteca está instalada, sem carregá-la
import json              # ler e gravar arquivos JSON
import os                # trocar arquivos de lugar com segurança
import re                # achar trechos de texto por padrão
import sys               # deixar o Python enxergar a pasta scripts/src
import time              # medir quanto tempo cada etapa leva
from pathlib import Path  # caminhos de pastas e arquivos

# Bibliotecas de cálculos e de tabelas (DataFrames).
import numpy as np
import pandas as pd

# Acha a raiz do projeto (a pasta do config.yaml), subindo a partir da pasta atual,
# e deixa o Python enxergar as funções comuns do projeto (scripts/src).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
if str(RAIZ / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ / "scripts"))

# Funções comuns: config.yaml, log, validação, arquivos de outras etapas.
from src import config, fontes, registro, validacao
from src.registro import formatar_tempo

# Lê o config.yaml e inicia o log (na tela e em 03_processamento/logs).
cfg = config.carregar(RAIZ)
log = registro.iniciar_log("03_carregar_bases_prefeitura", cfg)
# Hora de início do notebook (para o tempo total no resumo final).
inicio_notebook = time.time()
log.info("Preparando o notebook...")

# ===== PARÂMETROS (config.yaml) =====
CRS_SAIDA_EPSG = int(cfg.projeto("crs_graus").split(":")[1])     # 4674 = SIRGAS 2000 (lat/lon de saída)
CRS_METRICO_EPSG = int(cfg.projeto("crs_metrico").split(":")[1])  # 31983 = SIRGAS 2000 / UTM 23S (metros)
# SIRF
PASTA_SIRF = cfg.caminho("brutos.sirf")
PADRAO_SIRF = cfg.parametro("sirf.padrao")
SEP_SIRF = cfg.parametro("sirf.separador")
ENC_SIRF = cfg.parametro("sirf.codificacao")
COLUNAS_EXCLUIDAS_SIRF = cfg.parametro("sirf.colunas_excluidas")    # dados pessoais: nunca carregados
# Lotes e limite municipal (GeoPackages exportados do banco pelo QGIS)
ARQ_LOTES = cfg.caminho("brutos.lotes")
CAMADA_LOTES = cfg.camada("brutos.lotes")
GEO_COL_INSCRICAO = cfg.parametro("lotes.coluna_inscricao")         # coluna com a inscrição imobiliária
ARQ_LIMITE = cfg.caminho("brutos.limite_municipal")
CAMADA_LIMITE = cfg.camada("brutos.limite_municipal")
# Cadastro municipal
ARQ_CADASTRO = cfg.caminho("brutos.cadastro_municipal")
ABA_CADASTRO = cfg.parametro("cadastro.aba")
COLUNAS_CADASTRO = cfg.parametro("cadastro.colunas")                # título na planilha -> nome no código
COLUNAS_EXCLUIDAS_CADASTRO = cfg.parametro("cadastro.colunas_excluidas")  # dados pessoais

# ===== CACHE (Parquet + manifesto, na pasta 03_processamento/03_carregar_bases_prefeitura) =====
from src.cache import CacheTabelas, impressao_arquivos


def descrever_cache(nome):
    """Devolve (origem, parâmetros, dependências) de cada tabela: é isso que decide se o cache vale."""
    if nome == "df_sirf":
        return (impressao_arquivos(PASTA_SIRF, PADRAO_SIRF),
                {"separador": SEP_SIRF, "codificacao": ENC_SIRF, "colunas_excluidas": sorted(COLUNAS_EXCLUIDAS_SIRF)}, None)
    if nome in ("df_lotes", "df_lotes_descartados"):
        return (impressao_arquivos(ARQ_LOTES.parent, ARQ_LOTES.name),
                {"camada": CAMADA_LOTES, "coluna_inscricao": GEO_COL_INSCRICAO, "crs_saida_epsg": CRS_SAIDA_EPSG}, None)
    if nome == "df_limite":
        return (impressao_arquivos(ARQ_LIMITE.parent, ARQ_LIMITE.name),
                {"camada": CAMADA_LIMITE, "formato": "uma linha, desenho em WKB hexadecimal"}, None)
    if nome == "df_cadastro":
        return (impressao_arquivos(ARQ_CADASTRO.parent, ARQ_CADASTRO.name),
                {"aba": ABA_CADASTRO, "colunas": COLUNAS_CADASTRO,
                 "colunas_excluidas": sorted(COLUNAS_EXCLUIDAS_CADASTRO)}, None)
    raise KeyError(f"Tabela sem descrição de cache: {nome}")


cache = CacheTabelas(
    pasta=cfg.pasta_etapa("03_carregar_bases_prefeitura"),
    versoes=cfg.parametro("cache.versao_logica"),
    descrever=descrever_cache,
    forcar=cfg.parametro("cache.forcar_reprocessar", []),
    idade_maxima_dias=cfg.parametro("cache.idade_maxima_dias", {}),
    # Segurança: estas colunas (dados pessoais) nunca podem ir para o disco.
    colunas_proibidas={"df_sirf": COLUNAS_EXCLUIDAS_SIRF,
                       "df_cadastro": list(COLUNAS_EXCLUIDAS_CADASTRO) + ["cnpj_cpf"]},
)
# Mesmos nomes de antes, para as seções continuarem iguais.
verificar_cache, carregar_cache, salvar_cache = cache.verificar, cache.carregar, cache.salvar

# ===== ENTRADAS =====
log.info(f"SIRF: {len(impressao_arquivos(PASTA_SIRF, PADRAO_SIRF))} arquivo(s) em {PASTA_SIRF}")
for arquivo in (ARQ_LOTES, ARQ_LIMITE, ARQ_CADASTRO):
    log.info(f"{arquivo.name}: {'existe' if arquivo.is_file() else 'NÃO existe'}")
# Estabelecimentos da Receita (01): usados na conferência do cadastro municipal (seção 4).
df_estab = fontes.ler_processamento(cfg, "df_estab")

log.info(f"Preparação concluída! ({formatar_tempo(time.time() - inicio_notebook)})")

## 1. Carga do SIRF
Lê todos os arquivos CSV da pasta do SIRF (um por setor) e junta tudo em um único DataFrame, `df_sirf`.
As colunas com dados pessoais ("Código Pessoa", "Nome", "CPF" e "CNPJ") nunca são carregadas: são descartadas na própria leitura do arquivo, e o cache se recusa a gravar o SIRF se alguma delas aparecer.
Tudo é lido como texto, sem alterar nenhum valor. Acrescenta a coluna `arquivo_origem`. Se o cache vale, carrega dele. Não exibe linhas do DataFrame, só logs.

In [ ]:
log.info("SEÇÃO 1: SIRF (usa o cache quando ele vale)...")
inicio_b6 = time.time()

# Confere se o cache da tabela vale (arquivos de origem, parâmetros, versão e dependências iguais aos de quando foi gravado).
log.info("Conferindo o cache de df_sirf...")
usar_cache, motivo_cache = verificar_cache("df_sirf")
log.info("Conferência do cache concluída!")

if usar_cache:
    log.info(f"Usando o cache ({motivo_cache}): carregando df_sirf...")
    df_sirf = carregar_cache("df_sirf")
    log.info(f"df_sirf carregado do cache! {len(df_sirf):,} linhas | {df_sirf.shape[1]} colunas.")
else:
    log.info(f"Cache não usado ({motivo_cache}): lendo os arquivos originais...")
    import time
    from pathlib import Path
    import pandas as pd

    log.info("BLOCO SIRF: carregando os dados do SIRF...")
    # Guarda a hora de início para calcular o tempo total no final.
    inicio_sirf = time.time()

    # As configurações do SIRF (PASTA_SIRF, PADRAO_SIRF, SEP_SIRF, ENC_SIRF) vêm do config.yaml.
    # A lista de colunas com dados pessoais vem do config.yaml com o nome COLUNAS_EXCLUIDAS_SIRF;
    # aqui ela é guardada com o nome COLUNAS_EXCLUIDAS, que é o usado no resto deste bloco.
    COLUNAS_EXCLUIDAS = COLUNAS_EXCLUIDAS_SIRF

    # ===================================================================
    # ETAPA 1: PROCURAR OS ARQUIVOS
    # ===================================================================
    log.info("Etapa 1: procurando os arquivos na pasta...")

    # Lista os arquivos da pasta que combinam com o padrão, em ordem alfabética.
    arquivos_sirf = sorted(p for p in Path(PASTA_SIRF).glob(PADRAO_SIRF) if p.is_file())
    log.info(f"Etapa 1 concluída! {len(arquivos_sirf)} arquivo(s) encontrado(s).")

    # Se não achou nenhum, para aqui com uma mensagem.
    if not arquivos_sirf:
        raise FileNotFoundError("Nenhum arquivo encontrado. Confira PASTA_SIRF e PADRAO_SIRF.")

    # ===================================================================
    # ETAPA 2: LER OS ARQUIVOS
    # ===================================================================
    log.info("\nEtapa 2: lendo os arquivos (sem as colunas com dados pessoais)...")

    # Lista onde guardamos o conteúdo de cada arquivo.
    partes_sirf = []
    # Colunas que o primeiro arquivo trouxe (para conferir se os outros têm as mesmas).
    colunas_do_primeiro = None
    # Quais colunas excluídas realmente existiam nos arquivos (só para conferência).
    excluidas_encontradas = set()
    # Arquivos cujas colunas diferem das do primeiro.
    arquivos_com_colunas_diferentes = []

    # Passa por cada arquivo, um de cada vez (numero = 1, 2, 3...).
    for numero, arquivo in enumerate(arquivos_sirf, start=1):
        log.info(f"[{numero}/{len(arquivos_sirf)}] Lendo {arquivo.name}...")
        inicio_arquivo = time.time()

        # Lê SÓ o cabeçalho (nrows=0 significa "nenhuma linha de dados") para saber quais colunas pessoais existem.
        cabecalho = pd.read_csv(arquivo, sep=SEP_SIRF, encoding=ENC_SIRF, nrows=0).columns
        excluidas_encontradas.update(set(cabecalho) & set(COLUNAS_EXCLUIDAS))

        # Lê o arquivo inteiro:
        # dtype=str: tudo como texto, sem perder zeros à esquerda (inscrição, CEP);
        # usecols: só carrega as colunas que NÃO estão na lista de excluídas, então os dados pessoais nem entram na memória;
        # keep_default_na=False e na_values=[""]: só a célula realmente vazia vira "vazio"; textos como "NA" são mantidos.
        # Se alguma linha estiver malformada, o pandas para com um erro (e não pula a linha em silêncio).
        tabela_arquivo = pd.read_csv(
            arquivo,
            sep=SEP_SIRF,
            encoding=ENC_SIRF,
            dtype=str,
            usecols=lambda coluna: coluna not in COLUNAS_EXCLUIDAS,
            keep_default_na=False,
            na_values=[""],
        )

        # Guarda de qual arquivo veio cada linha.
        tabela_arquivo["arquivo_origem"] = arquivo.name

        # Confere se as colunas são as mesmas do primeiro arquivo.
        if colunas_do_primeiro is None:
            colunas_do_primeiro = list(tabela_arquivo.columns)
        elif list(tabela_arquivo.columns) != colunas_do_primeiro:
            arquivos_com_colunas_diferentes.append(arquivo.name)

        partes_sirf.append(tabela_arquivo)
        log.info(f"[{numero}/{len(arquivos_sirf)}] {arquivo.name} concluído! {len(tabela_arquivo):,} linhas "
              f"({formatar_tempo(time.time() - inicio_arquivo)}).")

    log.info("Etapa 2 concluída!")

    # ===================================================================
    # ETAPA 3: JUNTAR OS ARQUIVOS
    # ===================================================================
    log.info("\nEtapa 3: juntando os arquivos em um único DataFrame (df_sirf)...")

    # ignore_index=True refaz a numeração das linhas do zero (0, 1, 2...).
    df_sirf = pd.concat(partes_sirf, ignore_index=True)
    # Soma das linhas de cada arquivo, para conferir depois.
    soma_linhas_arquivos = sum(len(p) for p in partes_sirf)
    # Libera a memória das partes, que não são mais necessárias.
    del partes_sirf

    log.info(f"Etapa 3 concluída! df_sirf: {len(df_sirf):,} linhas | {df_sirf.shape[1]} colunas.")

    # ===================================================================
    # ETAPA 4: CONFERÊNCIAS
    # ===================================================================
    log.info("\nEtapa 4: conferindo o resultado...")

    # Colunas excluídas que, por algum motivo, ainda estão no DataFrame (não deveria haver nenhuma).
    sobraram = [c for c in COLUNAS_EXCLUIDAS if c in df_sirf.columns]
    # Colunas excluídas que não existiam nos arquivos (o nome pode estar diferente do esperado).
    nao_encontradas = [c for c in COLUNAS_EXCLUIDAS if c not in excluidas_encontradas]

    log.info("Etapa 4 concluída!")

    log.info("\n--- CONFERÊNCIAS ---")
    log.info(f"Linhas do df_sirf = soma das linhas dos arquivos ({soma_linhas_arquivos:,}): "
          f"{'confere' if len(df_sirf) == soma_linhas_arquivos else 'NÃO confere'}")
    log.info(f"Todos os arquivos com as mesmas colunas: "
          f"{'confere' if not arquivos_com_colunas_diferentes else 'NÃO confere: ' + str(arquivos_com_colunas_diferentes)}")
    log.info(f"Colunas com dados pessoais ausentes do df_sirf: "
          f"{'confere' if not sobraram else 'NÃO confere, sobraram: ' + str(sobraram)}")
    log.info(f"Colunas excluídas encontradas nos arquivos (para saber que o nome estava certo): "
          f"{sorted(excluidas_encontradas)}"
          + (f" | NÃO encontradas: {nao_encontradas}" if nao_encontradas else ""))

    log.info("\n--- RESUMO ---")
    log.info(f"df_sirf: {len(df_sirf):,} linhas | {df_sirf.shape[1]} colunas")
    log.info(f"Colunas carregadas (só os nomes; CONFIRA se alguma contém dado pessoal): {list(df_sirf.columns)}")

    log.info(f"\nBLOCO SIRF concluído! Tempo total: {formatar_tempo(time.time() - inicio_sirf)}.")

    # Grava o cache para as próximas sessões.
    salvar_cache("df_sirf", df_sirf)

log.info(f"\nSEÇÃO 1 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b6)}.")

## 2. Base de lotes georreferenciada
Lê os lotes de `01_dados_brutos/geosanja/lotes.gpkg` (camada de lotes exportada do banco pelo QGIS; ver `docs/fontes.md`): só a inscrição (`rotulo`) e o desenho. Gera um ponto garantido dentro de cada lote. Linhas sem inscrição, sem desenho ou com desenho inválido vão para `df_lotes_descartados`, com o motivo; se a mesma inscrição tiver mais de um desenho válido, fica o de maior área.
Se o cache de `df_lotes` e `df_lotes_descartados` vale (o GeoPackage não mudou), carrega dele. Para ler de novo, ponha `"df_lotes"` e `"df_lotes_descartados"` em `parametros.cache.forcar_reprocessar` (config.yaml).
Não exibe linhas dos DataFrames, só logs.

In [ ]:
log.info("SEÇÃO 2: lotes georreferenciados (usa o cache quando ele vale)...")
inicio_b7 = time.time()

# As duas tabelas desta seção precisam ter cache válido para o GeoPackage não ser lido de novo.
log.info("Conferindo o cache de df_lotes e df_lotes_descartados...")
usar_cache_lotes, motivo_cache_lotes = verificar_cache("df_lotes")
usar_cache_desc, motivo_cache_desc = verificar_cache("df_lotes_descartados")
log.info("Conferência do cache concluída!")

if usar_cache_lotes and usar_cache_desc:
    log.info(f"Usando o cache ({motivo_cache_lotes}): carregando df_lotes e df_lotes_descartados (o GeoPackage NÃO é lido)...")
    df_lotes = carregar_cache("df_lotes")
    df_lotes_descartados = carregar_cache("df_lotes_descartados")
    log.info(f"df_lotes carregado do cache! {len(df_lotes):,} linhas | df_lotes_descartados: {len(df_lotes_descartados):,} linhas.")
else:
    motivo = motivo_cache_lotes if not usar_cache_lotes else motivo_cache_desc
    log.info(f"Cache não usado ({motivo}): lendo os lotes do GeoPackage...")
    # Biblioteca para conferir se outras bibliotecas estão instaladas, sem carregá-las.
    import importlib.util

    log.info("Lendo a base de lotes georreferenciada...")
    # Guarda a hora de início para calcular o tempo total no final.
    inicio_lotes = time.time()

    # ===================================================================
    # ETAPA 1: VERIFICAR A BIBLIOTECA
    # ===================================================================
    log.info("Etapa 1: verificando a biblioteca geopandas...")
    # find_spec devolve None quando a biblioteca não está instalada.
    if importlib.util.find_spec("geopandas") is None:
        raise ImportError("Seção interrompida: falta a biblioteca geopandas. "
                          "Instale com: py -m pip install --user geopandas. Nada foi lido.")
    # Só importa depois de conferir que existe.
    import geopandas as gpd
    log.info("Etapa 1 concluída! Biblioteca encontrada.")

    # ===================================================================
    # ETAPA 2: LER OS LOTES DO GEOPACKAGE
    # ===================================================================
    log.info(f"\nEtapa 2: lendo os lotes de {ARQ_LOTES.name} (só a inscrição e o desenho)...")
    # O arquivo é a exportação da camada de lotes feita no QGIS (ver docs/fontes.md).
    if not ARQ_LOTES.is_file():
        raise FileNotFoundError(f"Seção interrompida: arquivo não encontrado: {ARQ_LOTES}. "
                                "Exporte a camada de lotes pelo QGIS para essa pasta (todas as feições, CRS original).")
    # Lê só a coluna da inscrição; o desenho vem sempre junto (coluna "geometry").
    lotes = gpd.read_file(ARQ_LOTES, layer=CAMADA_LOTES, columns=[GEO_COL_INSCRICAO])
    # Total de linhas lidas, usado na conferência final (todas as linhas precisam ter um destino).
    total_lido = len(lotes)

    # Se a camada veio vazia, para aqui.
    if lotes.empty:
        raise ValueError("Seção interrompida: a camada de lotes não tem nenhuma linha.")

    # O sistema de coordenadas de origem precisa estar definido; não vou chutar um.
    crs_origem = lotes.crs
    if crs_origem is None:
        raise ValueError("Seção interrompida: a camada de lotes não informa o sistema de coordenadas. "
                         "Informe o EPSG correto antes de continuar.")
    # A área dos desenhos (usada na Etapa 4) só faz sentido em um sistema de coordenadas em metros.
    if not crs_origem.is_projected:
        raise ValueError("Seção interrompida: o sistema de coordenadas de origem está em graus, "
                         "e a área dos lotes não seria confiável. Nada foi calculado.")

    log.info(f"Etapa 2 concluída! {len(lotes):,} linhas lidas. Sistema de coordenadas de origem: EPSG:{crs_origem.to_epsg()}.")

    # ===================================================================
    # ETAPA 3: CONFERIR OS DADOS LIDOS E SEPARAR OS DESCARTADOS
    # ===================================================================
    log.info("\nEtapa 3: conferindo os dados lidos e separando os descartados...")

    # Verdadeiro para quem não tem inscrição.
    sem_inscricao = lotes[GEO_COL_INSCRICAO].isna()
    # Verdadeiro para quem não tem desenho (vazio ou inexistente).
    sem_desenho = lotes.geometry.isna() | lotes.geometry.is_empty

    # Verdadeiro para quem tem desenho inválido (com defeito, como linhas que se cruzam).
    # Começa tudo falso e só confere os desenhos que existem.
    invalido = pd.Series(False, index=lotes.index)
    invalido.loc[~sem_desenho] = ~lotes.loc[~sem_desenho].geometry.is_valid

    # Motivo de cada linha descartada. Cada linha recebe UM motivo só: a primeira condição verdadeira vale.
    motivo = np.select(
        [sem_inscricao, sem_desenho, invalido],
        ["sem inscrição", "sem desenho", "desenho inválido"],
        default="",
    )
    # Verdadeiro para as linhas que saem do cálculo.
    descartada = motivo != ""

    # Guarda as linhas descartadas para você poder conferir depois (por exemplo, abrindo no QGIS).
    df_lotes_descartados = pd.DataFrame({
        "inscricao": lotes.loc[descartada, GEO_COL_INSCRICAO].to_numpy(),
        "motivo": motivo[descartada],
    })

    # Inscrições com espaço sobrando no começo ou no fim (poderiam atrapalhar o cruzamento depois).
    com_espaco = lotes[GEO_COL_INSCRICAO].dropna()
    com_espaco = int((com_espaco != com_espaco.str.strip()).sum())

    log.info("    Linhas descartadas (cada linha conta uma vez só):")
    for nome_motivo in ["sem inscrição", "sem desenho", "desenho inválido"]:
        log.info(f"        {nome_motivo}: {int((motivo == nome_motivo).sum()):,}")
    # Lista cada linha descartada com a inscrição e a coordenada (em graus) de um ponto dentro do desenho,
    # para você achar o lote no QGIS (colar "latitude, longitude" na barra de coordenadas).
    if descartada.any():
        log.info("    Descartadas (motivo | inscrição | latitude, longitude de um ponto do desenho), até 20:")
        tem_desenho = descartada & ~sem_desenho
        pontos_desc = gpd.GeoSeries(lotes.loc[tem_desenho].geometry.representative_point(), crs=crs_origem).to_crs(epsg=CRS_SAIDA_EPSG)
        for numero_linha in lotes.index[descartada][:20]:
            inscricao_desc = lotes.at[numero_linha, GEO_COL_INSCRICAO]
            if numero_linha in pontos_desc.index and not pontos_desc[numero_linha].is_empty:
                onde = f"{pontos_desc[numero_linha].y:.6f}, {pontos_desc[numero_linha].x:.6f}"
            else:
                onde = "(sem desenho)"
            log.info(f"        {motivo[lotes.index.get_loc(numero_linha)]} | {inscricao_desc if pd.notna(inscricao_desc) else '(vazia)'} | {onde}")
    log.info(f"    Inscrições com espaço sobrando nas pontas (mantidas como estão): {com_espaco:,}")

    # Fica só com as linhas que têm inscrição, têm desenho e o desenho é válido.
    lotes = lotes[~descartada]

    log.info(f"Etapa 3 concluída! {len(lotes):,} linhas seguem para o cálculo e {len(df_lotes_descartados):,} foram guardadas em df_lotes_descartados.")

    # ===================================================================
    # ETAPA 4: FICAR COM O MAIOR DESENHO DE CADA INSCRIÇÃO
    # ===================================================================
    log.info("\nEtapa 4: ficando com o maior desenho de cada inscrição...")

    # Quantos desenhos válidos cada inscrição tem.
    contagem = lotes[GEO_COL_INSCRICAO].value_counts()
    # Inscrições que aparecem em mais de uma linha.
    n_repetidas = int((contagem > 1).sum())
    log.info(f"    Inscrições distintas: {len(contagem):,} | com mais de um desenho: {n_repetidas:,}")

    # Área de cada desenho, em metros quadrados (o sistema de coordenadas de origem está em metros).
    lotes = lotes.assign(_area=lotes.geometry.area)
    # Ordena do maior para o menor desenho (mergesort mantém a ordem original nos empates).
    lotes = lotes.sort_values("_area", ascending=False, kind="mergesort")
    # Para cada inscrição, mantém só a primeira linha, que é a de maior área.
    linhas_antes = len(lotes)
    lotes = lotes.drop_duplicates(subset=GEO_COL_INSCRICAO, keep="first")
    descartados_menores = linhas_antes - len(lotes)

    # Uma coluna de desenhos, com uma linha por inscrição, e o sistema de coordenadas de origem.
    geometrias = gpd.GeoSeries(lotes.set_index(GEO_COL_INSCRICAO).geometry, crs=crs_origem)
    geometrias.index.name = "inscricao"

    log.info(f"    Desenhos menores descartados (de inscrições com mais de um desenho): {descartados_menores:,}")
    log.info(f"Etapa 4 concluída! {len(geometrias):,} desenhos, um por inscrição.")

    # ===================================================================
    # ETAPA 5: PONTO GARANTIDO DENTRO DE CADA LOTE
    # ===================================================================
    log.info("\nEtapa 5: calculando o ponto dentro de cada lote...")

    # O ponto representativo é calculado no sistema de coordenadas de origem (onde as distâncias são corretas)
    # e sempre cai dentro do desenho, mesmo em lotes em formato de L ou de U.
    pontos_origem = geometrias.representative_point()

    # Confere se cada ponto está mesmo dentro do seu lote.
    dentro = pontos_origem.within(geometrias)
    fora = int((~dentro).sum())

    # Converte os pontos para o sistema de coordenadas de saída (o mesmo do CNEFE).
    pontos_saida = pontos_origem.to_crs(epsg=CRS_SAIDA_EPSG)

    # Lista as inscrições cujo ponto ficou fora do próprio lote (para conferir no QGIS).
    if fora:
        log.info("    Pontos fora do seu lote (inscrição | latitude, longitude do ponto):")
        for inscricao_fora, ponto_fora in pontos_saida[~dentro.to_numpy()].items():
            log.info(f"        {inscricao_fora} | {ponto_fora.y:.6f}, {ponto_fora.x:.6f}")

    log.info(f"Etapa 5 concluída! {len(pontos_saida):,} pontos calculados.")

    # ===================================================================
    # ETAPA 6: MONTAR O df_lotes
    # ===================================================================
    log.info("\nEtapa 6: montando o df_lotes...")

    # Tabela final: uma linha por inscrição, com latitude e longitude em graus decimais (sem arredondar).
    # No ponto, x é a longitude e y é a latitude.
    df_lotes = pd.DataFrame({
        "inscricao": pontos_saida.index.to_numpy(),
        "lat": pontos_saida.y.to_numpy(),
        "lon": pontos_saida.x.to_numpy(),
        # Quantos desenhos válidos a inscrição tinha na tabela (1 na maioria; mais de 1 quando se repete).
        "n_poligonos": contagem.reindex(pontos_saida.index).to_numpy(),
    })

    # Nos descartados, marca se a inscrição acabou resolvida por outro desenho válido dela.
    # Assim você vê quais descartados deixaram a inscrição sem ponto de verdade.
    df_lotes_descartados["inscricao_tem_ponto"] = df_lotes_descartados["inscricao"].isin(df_lotes["inscricao"])

    log.info(f"Etapa 6 concluída! df_lotes: {len(df_lotes):,} linhas | {df_lotes.shape[1]} colunas.")

    # ===================================================================
    # ETAPA 7: CONFERÊNCIAS E RESUMO
    # ===================================================================
    log.info("\nEtapa 7: conferindo o resultado...")

    log.info("\n--- CONFERÊNCIAS ---")
    log.info(f"Inscrições todas diferentes no df_lotes: {'confere' if df_lotes['inscricao'].is_unique else 'NÃO confere'}")
    log.info(f"Linhas do df_lotes = inscrições distintas com desenho válido ({len(contagem):,}): "
          f"{'confere' if len(df_lotes) == len(contagem) else 'NÃO confere'}")
    # Todas as linhas lidas precisam ter ido para algum lugar: virou ponto, foi descartada ou era desenho menor.
    soma_destinos = len(df_lotes) + len(df_lotes_descartados) + descartados_menores
    log.info(f"Linhas lidas = df_lotes + descartados + desenhos menores ({total_lido:,}): "
          f"{'confere' if soma_destinos == total_lido else 'NÃO confere'}")
    log.info(f"Pontos fora do seu lote: {fora:,} ({'confere' if fora == 0 else 'NÃO confere'})")
    log.info(f"Latitude e longitude sem vazios: "
          f"{'confere' if df_lotes[['lat', 'lon']].notna().all().all() else 'NÃO confere'}")

    log.info("\n--- RESUMO ---")
    log.info(f"df_lotes: {len(df_lotes):,} linhas | colunas: {list(df_lotes.columns)}")
    log.info(f"df_lotes_descartados: {len(df_lotes_descartados):,} linhas | colunas: {list(df_lotes_descartados.columns)}")
    # Descartados com inscrição que continuaram sem ponto (a inscrição não tem nenhum desenho válido).
    sem_ponto = df_lotes_descartados[df_lotes_descartados["inscricao"].notna() & ~df_lotes_descartados["inscricao_tem_ponto"]]
    log.info(f"  Descartados com inscrição que ficaram SEM ponto no df_lotes: {len(sem_ponto):,}")
    log.info(f"Sistema de coordenadas de saída: EPSG:{CRS_SAIDA_EPSG}")
    log.info(f"Latitude: de {df_lotes['lat'].min():.6f} a {df_lotes['lat'].max():.6f}")
    log.info(f"Longitude: de {df_lotes['lon'].min():.6f} a {df_lotes['lon'].max():.6f}")
    log.info(f"Inscrições com mais de um desenho válido (ficou o maior): {int((df_lotes['n_poligonos'] > 1).sum()):,}")

    log.info(f"\nLeitura dos lotes concluída! Tempo total: {formatar_tempo(time.time() - inicio_lotes)}.")

    # Grava o cache das duas tabelas para as próximas sessões.
    salvar_cache("df_lotes", df_lotes)
    salvar_cache("df_lotes_descartados", df_lotes_descartados)

log.info(f"\nSEÇÃO 2 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b7)}.")

## 3. Limite municipal
Lê o limite do município de `01_dados_brutos/geosanja/limite_municipal.gpkg` (camada do limite municipal exportada do banco pelo QGIS) e monta `df_limite`, com **uma linha**: o desenho do limite em texto (WKB hexadecimal), o código EPSG de origem e a área em km².
Se o cache de `df_limite` vale (o GeoPackage não mudou), carrega dele. É usado no notebook 07 (seção 4), que garante que todos os pontos fiquem dentro do município.
Não exibe linhas do DataFrame, só logs.

In [ ]:
log.info("SEÇÃO 3: limite municipal (usa o cache quando ele vale)...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_b8 = time.time()

# Confere se o cache da tabela vale (origem, parâmetros e versão iguais aos de quando foi gravado).
log.info("Conferindo o cache de df_limite...")
usar_cache, motivo_cache = verificar_cache("df_limite")
log.info("Conferência do cache concluída!")

if usar_cache:
    log.info(f"Usando o cache ({motivo_cache}): carregando df_limite (o GeoPackage NÃO é lido)...")
    df_limite = carregar_cache("df_limite")
    log.info(f"df_limite carregado do cache! {len(df_limite):,} linha | área: {df_limite.loc[0, 'area_km2']:,.1f} km² | "
          f"EPSG de origem: {df_limite.loc[0, 'epsg']}.")
else:
    log.info(f"Cache não usado ({motivo_cache}): lendo o limite municipal do GeoPackage...")

    # ===================================================================
    # ETAPA 1: VERIFICAR AS BIBLIOTECAS
    # ===================================================================
    log.info("\nEtapa 1: verificando as bibliotecas...")
    # Lista das bibliotecas que faltam (começa vazia).
    faltam = []
    for nome_biblioteca in ["geopandas", "shapely"]:
        # find_spec devolve None quando a biblioteca não está instalada.
        if importlib.util.find_spec(nome_biblioteca) is None:
            faltam.append(nome_biblioteca)
    if faltam:
        raise ImportError("Seção interrompida: faltam as bibliotecas " + ", ".join(faltam) + ". "
                          "Instale com: py -m pip install --user geopandas shapely. Nada foi lido.")
    # Só importa depois de conferir que existem.
    import geopandas as gpd
    log.info("Etapa 1 concluída! Bibliotecas encontradas.")

    # ===================================================================
    # ETAPA 2: LER A CAMADA DO LIMITE MUNICIPAL
    # ===================================================================
    log.info(f"\nEtapa 2: lendo a camada do limite municipal ({ARQ_LIMITE.name})...")
    # O arquivo é a exportação da camada do limite municipal feita no QGIS (ver docs/fontes.md).
    if not ARQ_LIMITE.is_file():
        raise FileNotFoundError(f"Seção interrompida: arquivo não encontrado: {ARQ_LIMITE}. "
                                "Exporte a camada do limite municipal pelo QGIS para essa pasta.")
    # Lê a camada inteira (é pequena): o desenho vem na coluna "geometry".
    limite = gpd.read_file(ARQ_LIMITE, layer=CAMADA_LIMITE)

    # O sistema de coordenadas precisa estar definido e ter código EPSG; não vou chutar um.
    if limite.crs is None or limite.crs.to_epsg() is None:
        raise ValueError("Bloco interrompido: a camada do limite não informa um sistema de coordenadas com código EPSG.")
    epsg_limite = int(limite.crs.to_epsg())
    # Padroniza o nome da coluna do desenho e descarta linhas sem desenho.
    if limite.geometry.name != "geometry":
        limite = limite.rename_geometry("geometry")
    vazias = int((limite.geometry.isna() | limite.geometry.is_empty).sum())
    limite = limite[~(limite.geometry.isna() | limite.geometry.is_empty)]
    if limite.empty:
        raise ValueError("Bloco interrompido: a camada do limite não tem nenhum desenho.")
    # Conserta desenhos inválidos (ex.: linhas que se cruzam), se houver.
    invalidas = int((~limite.geometry.is_valid).sum())
    if invalidas:
        limite["geometry"] = limite.geometry.make_valid()
    log.info(f"Etapa 2 concluída! {len(limite):,} feição(ões) | EPSG:{epsg_limite} | vazias descartadas: {vazias} | "
          f"inválidas consertadas: {invalidas}.")

    # ===================================================================
    # ETAPA 3: MONTAR O df_limite (UMA LINHA, COM O DESENHO EM TEXTO)
    # ===================================================================
    log.info("\nEtapa 3: montando o df_limite...")
    # Junta todas as feições num desenho só (union_all nas versões novas do geopandas, unary_union nas antigas).
    if hasattr(limite.geometry, "union_all"):
        uniao = limite.geometry.union_all()
    else:
        uniao = limite.geometry.unary_union
    # Área em km², medida em metros (SIRGAS 2000 / UTM 23S), só para conferir se faz sentido.
    area_km2 = float(gpd.GeoSeries([uniao], crs=epsg_limite).to_crs(CRS_METRICO_EPSG).iloc[0].area / 1e6)
    # O desenho vai para o cache como texto (WKB em hexadecimal), junto com o EPSG de origem.
    df_limite = pd.DataFrame({
        "geometria_wkb_hex": [uniao.wkb_hex],
        "epsg": [epsg_limite],
        "n_feicoes": [int(len(limite))],
        "area_km2": [area_km2],
    })
    log.info(f"Etapa 3 concluída! Área do limite: {area_km2:,.1f} km² (confira se faz sentido para o município).")

    # Grava o cache para as próximas sessões (o GeoPackage só é lido de novo se mudar).
    salvar_cache("df_limite", df_limite)

log.info(f"\nSEÇÃO 3 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b8)}.")

## 4. Cadastro municipal (cadastro v2.xlsx)
Lê a planilha do cadastro municipal (`brutos.cadastro_municipal`, no config.yaml) e monta `df_cadastro`, com **uma linha por inscrição municipal de CNPJ**. Linhas de CPF e a coluna do nome do contribuinte são descartadas na própria leitura (como no SIRF): o cache e o `df_cadastro` nunca têm CPF nem nome.
Ficam o CNPJ de 14 dígitos (a chave do cruzamento), a inscrição municipal, endereço, rua, bairro, CEP, cidade, o CNAE principal (só dígitos) e sua descrição, e os campos `estabelecido`, `tipo` e `reg_tributario`. Se o cache vale, carrega dele.
Depois da leitura, mostra um **diagnóstico** (só contagens e "máscaras" de formato, nenhuma linha de dados): quantos CNPJs, duplicados, quantos cruzam com a Receita, onde o CNAE diverge e como são escritos o endereço, a rua e o CEP. Você pode apagar a parte do diagnóstico depois.
Não altera nenhuma outra base.

In [ ]:
# Biblioteca para tirar acentos de textos (só para comparar os títulos das colunas da planilha).
import unicodedata

log.info("SEÇÃO 4: cadastro municipal (usa o cache quando ele vale)...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_cadastro = time.time()

# Confere se o cache da tabela vale (arquivo de origem, parâmetros e versão iguais aos de quando foi gravado).
log.info("Conferindo o cache de df_cadastro...")
usar_cache, motivo_cache = verificar_cache("df_cadastro")
log.info("Conferência do cache concluída!")

if usar_cache:
    log.info(f"Usando o cache ({motivo_cache}): carregando df_cadastro...")
    df_cadastro = carregar_cache("df_cadastro")
    log.info(f"df_cadastro carregado do cache! {len(df_cadastro):,} linhas | {df_cadastro.shape[1]} colunas.")
else:
    log.info(f"Cache não usado ({motivo_cache}): lendo a planilha do cadastro municipal...")

    # ===================================================================
    # ETAPA 1: LER A PLANILHA
    # ===================================================================
    log.info("\nEtapa 1: lendo a planilha (pode demorar, o Excel é lento de ler)...")
    # Se o arquivo não existe, para aqui, sem ler nada.
    if not Path(ARQ_CADASTRO).is_file():
        raise FileNotFoundError(f"Bloco interrompido: arquivo não encontrado: {ARQ_CADASTRO}. "
                                "Ajuste ARQ_CADASTRO no config.yaml. Nada foi lido.")
    # dtype=str: lê tudo como texto, sem perder zeros à esquerda nem mudar o formato de números e datas.
    try:
        bruto = pd.read_excel(ARQ_CADASTRO, sheet_name=ABA_CADASTRO, dtype=str)
    except ImportError:
        raise ImportError("Bloco interrompido: falta a biblioteca openpyxl para ler .xlsx "
                          "(py -m pip install --user openpyxl). Nada foi lido.")
    log.info(f"Etapa 1 concluída! {len(bruto):,} linhas e {bruto.shape[1]} colunas lidas.")

    # ===================================================================
    # ETAPA 2: CONFERIR OS TÍTULOS DAS COLUNAS E RENOMEAR
    # ===================================================================
    log.info("\nEtapa 2: conferindo os títulos das colunas...")
    # Compara os títulos sem olhar espaços, maiúsculas, minúsculas e acentos (ex.: "Endereço" = "ENDERECO ").
    # (Esta função é própria do bloco: as funções de limpeza só são definidas depois, no src/enderecos.py.)
    def chave_titulo(texto):
        """Título em maiúsculas, sem acentos e com espaços únicos (só para comparar)."""
        # NFKD separa as letras dos acentos; encode/decode em ASCII descarta os acentos.
        sem_acento = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii")
        return " ".join(sem_acento.upper().split())

    titulos_planilha = {chave_titulo(c): c for c in bruto.columns}
    # Para cada coluna esperada, acha o título correspondente na planilha.
    renomear = {}
    faltando = []
    for titulo_esperado, nome_interno in COLUNAS_CADASTRO.items():
        achado = titulos_planilha.get(chave_titulo(titulo_esperado))
        if achado is None:
            faltando.append(titulo_esperado)
        else:
            renomear[achado] = nome_interno
    # Se faltar alguma, para aqui e mostra os títulos que a planilha tem (títulos não são dados pessoais).
    if faltando:
        raise KeyError("Bloco interrompido: títulos não encontrados na planilha: " + ", ".join(faltando) +
                       ". Títulos que a planilha tem: " + ", ".join(str(c) for c in bruto.columns) +
                       ". Ajuste COLUNAS_CADASTRO no config.yaml.")
    # Fica só com as colunas esperadas, já com o nome interno.
    bruto = bruto[list(renomear)].rename(columns=renomear)
    log.info("Etapa 2 concluída! Todas as colunas esperadas encontradas.")

    # ===================================================================
    # ETAPA 3: DESCARTAR CPF E DADOS PESSOAIS
    # ===================================================================
    log.info("\nEtapa 3: separando CNPJ de CPF e descartando os dados pessoais...")
    # Só os dígitos do documento (tira pontos, barra e traço).
    digitos = bruto["cnpj_cpf"].fillna("").astype(str).str.replace(r"\D", "", regex=True)
    tamanho = digitos.str.len()
    # CNPJ: 14 dígitos (o Excel pode ter comido zeros à esquerda: 12 e 13 dígitos são completados com zeros).
    eh_cnpj = tamanho.between(12, 14)
    # CPF: até 11 dígitos (e não vazio). Qualquer outra coisa é "documento fora do padrão".
    eh_cpf = tamanho.between(1, 11)
    vazio = tamanho == 0
    fora_padrao = ~eh_cnpj & ~eh_cpf & ~vazio
    log.info(f"    CNPJ: {int(eh_cnpj.sum()):,} | CPF (descartados): {int(eh_cpf.sum()):,} | "
          f"sem documento (descartados): {int(vazio.sum()):,} | fora do padrão (descartados): {int(fora_padrao.sum()):,}")

    # Fica só com as linhas de CNPJ. A coluna do nome e a do documento original NÃO seguem adiante.
    cad = bruto[eh_cnpj].copy()
    cad["cnpj"] = digitos[eh_cnpj].str.zfill(14)
    cad = cad.drop(columns=["cnpj_cpf"] + [c for c in COLUNAS_EXCLUIDAS_CADASTRO if c in cad.columns])
    del bruto, digitos, tamanho
    log.info(f"Etapa 3 concluída! {len(cad):,} linhas de CNPJ seguem.")

    # ===================================================================
    # ETAPA 4: LIMPAR OS CAMPOS
    # ===================================================================
    log.info("\nEtapa 4: limpando os campos...")
    # Texto sem espaços nas pontas; vazio vira "sem valor" (NaN).
    for coluna in [c for c in cad.columns if c != "cnpj"]:
        texto = cad[coluna].str.strip()
        cad[coluna] = texto.mask(texto == "")
    # CEP e CNAE: só os dígitos (tira pontos, traço e barra). Vazio continua vazio.
    for coluna in ["cep", "cnae_principal"]:
        so_digitos = cad[coluna].str.replace(r"\D", "", regex=True)
        cad[coluna] = so_digitos.mask(so_digitos == "")
    # Ordem das colunas: o CNPJ primeiro.
    cad = cad[["cnpj"] + [c for c in cad.columns if c != "cnpj"]]
    # Numeração das linhas 0, 1, 2... (o cache exige).
    df_cadastro = cad.reset_index(drop=True)
    del cad
    log.info(f"Etapa 4 concluída! df_cadastro: {len(df_cadastro):,} linhas | colunas: {list(df_cadastro.columns)}")

    # Grava o cache para as próximas sessões.
    salvar_cache("df_cadastro", df_cadastro)

# ===================================================================
# DIAGNÓSTICO (só contagens e formatos; pode ser apagado depois)
# ===================================================================
log.info("\nDiagnóstico do cadastro municipal...")

# CNPJ de 14 dígitos de cada estabelecimento da Receita (para o cruzamento).
cnpj_receita = df_estab["cnpj_basico"] + df_estab["cnpj_ordem"] + df_estab["cnpj_dv"]
total_cad = len(df_cadastro)
distintos = int(df_cadastro["cnpj"].nunique())
repetidos = df_cadastro["cnpj"].value_counts()
log.info(f"    Linhas: {total_cad:,} | CNPJs distintos: {distintos:,}")
log.info(f"    CNPJs com mais de uma inscrição: {int((repetidos > 1).sum()):,} (máximo de inscrições em um CNPJ: {int(repetidos.max())})")

# Cruzamento com a Receita.
cad_na_receita = df_cadastro["cnpj"].isin(set(cnpj_receita))
log.info(f"    CNPJs do cadastro que existem na base da Receita: {int(cad_na_receita.sum()):,} de {total_cad:,} linhas "
      f"({int((~cad_na_receita).sum()):,} não existem)")
receita_no_cad = cnpj_receita.isin(set(df_cadastro["cnpj"]))
log.info(f"    Estabelecimentos da Receita com cadastro municipal: {int(receita_no_cad.sum()):,} de {len(cnpj_receita):,} "
      f"({receita_no_cad.mean() * 100:.1f}%)")
log.info("    Cobertura por situação cadastral da Receita (com cadastro / total):")
sit_num_cad = pd.to_numeric(df_estab["situacao_cadastral"], errors="coerce")
for codigo_sit, nome_sit in {1: "Nula", 2: "Ativa", 3: "Suspensa", 4: "Inapta", 8: "Baixada"}.items():
    do_grupo = sit_num_cad == codigo_sit
    if do_grupo.any():
        log.info(f"        {nome_sit}: {int((do_grupo & receita_no_cad).sum()):,} de {int(do_grupo.sum()):,} "
              f"({(do_grupo & receita_no_cad).sum() / do_grupo.sum() * 100:.1f}%)")

# CNAE e CEP do cadastro comparados com os da Receita (só onde os dois existem; usa a primeira inscrição de cada CNPJ).
primeira = df_cadastro.drop_duplicates("cnpj").set_index("cnpj")
receita_idx = pd.DataFrame({"cnpj": cnpj_receita.values, "cnae_rf": df_estab["cnae_principal"].values,
                            "cep_rf": df_estab["cep"].fillna("").astype(str).str.strip().values}).drop_duplicates("cnpj").set_index("cnpj")
junto = primeira.join(receita_idx, how="inner")
if len(junto):
    cnae_ambos = junto["cnae_principal"].notna() & junto["cnae_rf"].notna()
    log.info(f"    CNAE principal: iguais {int((cnae_ambos & (junto['cnae_principal'] == junto['cnae_rf'])).sum()):,} | "
          f"diferentes {int((cnae_ambos & (junto['cnae_principal'] != junto['cnae_rf'])).sum()):,} | "
          f"sem CNAE no cadastro {int(junto['cnae_principal'].isna().sum()):,}")
    cep_ambos = junto["cep"].notna() & (junto["cep_rf"] != "")
    log.info(f"    CEP: iguais {int((cep_ambos & (junto['cep'] == junto['cep_rf'])).sum()):,} | "
          f"diferentes {int((cep_ambos & (junto['cep'] != junto['cep_rf'])).sum()):,} | "
          f"sem CEP no cadastro {int(junto['cep'].isna().sum()):,}")


def mascara_formato(serie):
    """Troca letras por 'A' e números por '9' (mantém pontuação): mostra o FORMATO sem mostrar o dado."""
    return (serie.dropna().astype(str).str.replace(r"[^\W\d_]+", "A", regex=True).str.replace(r"\d+", "9", regex=True))


log.info("    Cidade (valores mais frequentes):")
for valor, qtd in df_cadastro["cidade"].value_counts().head(5).items():
    log.info(f"        {valor}: {qtd:,}")
for coluna in ["endereco", "rua", "bairro", "cep", "estabelecido", "tipo", "reg_tributario"]:
    preenchidos = int(df_cadastro[coluna].notna().sum())
    distintos_col = int(df_cadastro[coluna].nunique())
    log.info(f"    {coluna}: preenchidos {preenchidos:,} | valores distintos {distintos_col:,}")
    if coluna in ["tipo", "reg_tributario", "estabelecido"] and distintos_col <= 30:
        # Poucos valores: mostra todos (são categorias, não dados pessoais).
        for valor, qtd in df_cadastro[coluna].value_counts().items():
            log.info(f"        '{valor}': {qtd:,}")
    elif coluna in ["endereco", "rua", "cep", "estabelecido"]:
        # Muitos valores: mostra só os formatos mais comuns.
        for formato, qtd in mascara_formato(df_cadastro[coluna]).value_counts().head(6).items():
            log.info(f"        formato '{formato}': {qtd:,}")
log.info("Diagnóstico concluído!")

log.info(f"\nSEÇÃO 4 concluída! Tempo total: {formatar_tempo(time.time() - inicio_cadastro)}.")

## 5. Validação
Linhas, nulos e duplicados de cada tabela, limite com uma linha só e nenhuma coluna de dados pessoais.

In [ ]:
log.info("SEÇÃO 5: validação das tabelas...")
validacao.validar_tabela("df_sirf", df_sirf, obrigatorias=[cfg.parametro("sirf.col_inscricao")])
validacao.validar_tabela("df_lotes", df_lotes, chave="inscricao", obrigatorias=["inscricao", "lat", "lon"])
validacao.validar_tabela("df_cadastro", df_cadastro, obrigatorias=["cnpj"])
# O limite municipal precisa ser uma linha só, com o desenho.
assert len(df_limite) == 1 and df_limite["geometria_wkb_hex"].notna().all(), "df_limite precisa ter uma linha com o desenho"
# Segurança: nenhuma coluna de dados pessoais pode ter sobrado.
pessoais = [c for c in COLUNAS_EXCLUIDAS_SIRF if c in df_sirf.columns] + \
           [c for c in list(COLUNAS_EXCLUIDAS_CADASTRO) + ["cnpj_cpf"] if c in df_cadastro.columns]
assert not pessoais, f"Colunas com dados pessoais encontradas: {pessoais}"
log.info("Sem colunas de dados pessoais no SIRF e no cadastro: confere!")
log.info("SEÇÃO 5 concluída!")

## 6. Resumo
Números principais, arquivos gerados e tempo total.

In [ ]:
log.info("RESUMO do 03_carregar_bases_prefeitura...")
log.info(f"SIRF: {len(df_sirf):,} | lotes: {len(df_lotes):,} (descartados: {len(df_lotes_descartados):,}) | cadastro: {len(df_cadastro):,}")
log.info("Arquivos gerados:")
for nome_tabela in ['df_sirf', 'df_lotes', 'df_lotes_descartados', 'df_limite', 'df_cadastro']:
    caminho = cfg.caminho("processamento." + nome_tabela)
    log.info(f"    {caminho.relative_to(RAIZ)} ({caminho.stat().st_size / 1e6:.1f} MB)")
log.info(f"RESUMO concluído! Tempo total do notebook: {formatar_tempo(time.time() - inicio_notebook)}.")